In [1]:
!pip install pystac-client rasterio geopandas  pyproj matplotlib numpy pandas
!pip install rioxarray
!pip install --upgrade stackstac
!pip install planetary-computer

  Using cached pystac_client-0.9.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached matplotlib-3.10.9-cp312-cp312-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached pyogrio-0.12.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (5.9 kB)
  Using cached shapely-2.1.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.1 kB)
Using cached pystac_client-0.9.0-py3-none-any.whl (41 kB)
Using cached matplotlib-3.10.9-cp312-cp312-macosx_11_0_arm64.whl (8.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 2.9 MB/s eta 0:00:0000:0100:01
Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl (273 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 3.0 MB/s eta 0:00:00a 0:00:01
Using cached kiwisolver-

In [2]:
from pystac_client import Client
import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray
import xarray as xr
import io
import pandas as pd
import stackstac
import rasterio
from typing import Callable, Dict, List, Optional, Union
import planetary_computer as pc

In [3]:
# itemediate function to handle data format
def handle_feature_collection(feature_collection):
  type_location_geojson = feature_collection["type"]
  if type_location_geojson not in ["Point", "Polygon", "feature", "FeatureCollection"]:

              raise ValueError("Only Point and Polygon types are supported.")

  if type_location_geojson == "FeatureCollection":
          return feature_collection["features"][0]["geometry"]

In [4]:
def get_stack_items(stac_url:str,
                    collection:str,
                    datetime:str,
                    feature_collection: str,
                    cloud_cover:int = 10,
                    planet_comp:bool = True
                    ):

  client = Client.open(stac_url)
  feature_collection = handle_feature_collection(feature_collection)
  search = client.search(
    collections=[collection],
    intersects=feature_collection,
    datetime=datetime,
    query=[f"eo:cloud_cover<{cloud_cover}"],
    limit=10
    )
  if planet_comp:
    items = pc.sign(search)
  else:
    items = search.item_collection()

  print(f"Found {len(items)} items")

  return items

In [5]:
# Define a GeoJSON feature collection for the search
feature_collection = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "coordinates": [
          [
            [
              35.091792388971584,
              0.5900286891046989
            ],
            [
              35.091792388971584,
              0.5741557135470998
            ],
            [
              35.123369292604906,
              0.5741557135470998
            ],
            [
              35.123369292604906,
              0.5900286891046989
            ],
            [
              35.091792388971584,
              0.5900286891046989
            ]
          ]
        ],
        "type": "Polygon"
      }
    }
  ]
}

gdf = gpd.GeoDataFrame.from_features(feature_collection["features"])
bbox = gdf.total_bounds

In [6]:
stac_url_local= "http://localhost:8081"

In [15]:
drone_items = get_stack_items(stac_url_local,'test-data',
                                  "2000-01-01/2026-06-30", feature_collection,
                                    planet_comp= False
                                  )

Found 0 items


In [36]:
import pystac_client
import rioxarray
# import s3fs

# Connect
catalog = pystac_client.Client.open("http://e-safari.acmad.org:8081")

# Search — no datetime/bbox filters, just the collection
results = catalog.search(collections=["monthly-precipitation"])
items = list(results.items())

print(f"Found {len(items)} items")

# Inspect the first item
item = items[0]
print("ID:      ", item.id)
print("Datetime:", item.datetime)
print("BBox:    ", item.bbox)
print("Assets:  ", list(item.assets.keys()))

href = item.assets["data"].href
print("HREF:    ", href)




Found 1 items
ID:       monthly-precipitation_2023-11-30T0000000000
Datetime: 2023-11-30 00:00:00+00:00
BBox:     [-25.5, -40.5, 60.5, 40.5]
Assets:   ['data']
HREF:     s3://string/monthly-precipitation/2026/02/314b8f57-0e5e-4671-be03-2d57ee4e12c7-AFR_Nov_2023_CPC-UNI_Percentile.tif
